# Ollama with Python

- [ollama.com](https://ollama.com/)
- [ollama.com/models](https://ollama.com/models)

In [1]:
%%time
import warnings
warnings.filterwarnings("ignore")

!python3 -m pip install --upgrade pip

CPU times: user 6.89 ms, sys: 8.09 ms, total: 15 ms
Wall time: 1 s


In [2]:
%pip install -qU ollama asyncio ipython-autotime --prefer-binary

%load_ext autotime

Note: you may need to restart the kernel to use updated packages.
time: 156 μs (started: 2025-06-22 14:41:35 -07:00)


## Define varibles

In [3]:
my_model='llama3.2'
my_sentence='What is three minus one?'

print(f"Model: {my_model} ; sentence: {my_sentence} ")

Model: llama3.2 ; sentence: What is three minus one? 
time: 440 μs (started: 2025-06-22 14:41:35 -07:00)


##  Call a function with a model

In [4]:
from ollama import ChatResponse, chat


def add_two_numbers(a: int, b: int) -> int:
  return int(a) + int(b)  # "what is 30 + 12" to produce '3012' instead of 42


def subtract_two_numbers(a: int, b: int) -> int:
  return int(a) - int(b)


# Tools can still be manually defined and passed into chat
subtract_two_numbers_tool = {
  'type': 'function',
  'function': {
    'name': 'subtract_two_numbers',
    'description': 'Subtract two numbers',
    'parameters': {
      'type': 'object',
      'required': ['a', 'b'],
      'properties': {
        'a': {'type': 'integer', 'description': 'The first number'},
        'b': {'type': 'integer', 'description': 'The second number'},
      },
    },
  },
}


messages = [{'role': 'user', 'content': my_sentence}]
print('Prompt:', messages[0]['content'])

available_functions = {
  'add_two_numbers': add_two_numbers,
  'subtract_two_numbers': subtract_two_numbers,
}

response: ChatResponse = chat(
  my_model,
  messages=messages,
  tools=[add_two_numbers, subtract_two_numbers_tool],
)

if response.message.tool_calls:
  # There may be multiple tool calls in the response
  for tool in response.message.tool_calls:
    # Ensure the function is available, and then call it
    if function_to_call := available_functions.get(tool.function.name):
      print('Calling function:', tool.function.name)
      print('Arguments:', tool.function.arguments)
      output = function_to_call(**tool.function.arguments)
      print('Function output:', output)
    else:
      print('Function', tool.function.name, 'not found')

# Only needed to chat with the model using the tool call results
if response.message.tool_calls:
  # Add the function response to messages for the model to use
  messages.append(response.message)
  messages.append({'role': 'tool', 'content': str(output), 'name': tool.function.name})

  # Get final response from model with function outputs
  final_response = chat(my_model, messages=messages)
  print('Final response:', final_response.message.content)

else:
  print('No tool calls returned from model')

Prompt: What is three minus one?
Calling function: subtract_two_numbers
Arguments: {'a': '3', 'b': '1'}
Function output: 2
Final response: Three minus one equals two.
time: 1.49 s (started: 2025-06-22 14:41:35 -07:00)
